In [1]:
print("hi")

hi


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline

from scikeras.wrappers import KerasClassifier
import tensorflow as tf
import pickle

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


In [6]:
data=pd.read_csv('Churn.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [7]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model



In [8]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [9]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [10]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which poi

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7727 - loss: 0.5021    
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6879 - loss: 0.5788
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7759 - loss: 0.4911
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6885 - loss: 0.6085
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5477 - loss: 0.7121
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7814 - loss: 0.5075
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7941 - loss: 0.4709
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7576 - loss: 0.5321
Epoch 2/50
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7902 - loss: 0.4784
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7559 - loss: 

/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/keras/src/layers/cor

167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 598us/step - accuracy: 0.8661 - loss: 0.3152
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 647us/step
83/84 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/stepEpoch 1/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 901us/stepep - accuracy: 0.8709 - loss: 0.31
137/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8714 - loss: 0.3170  Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8669 - loss: 0.3206
Epoch 1/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 634us/step
Epoch 1/50
Epoch 1/50
Epoch 1/50
 1/84 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/stepEpoch 1/100
Epoch 1/50
Epoch 1/100


/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/keras/src/layers/cor

... (23 outputs left out)

/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/keras/src/layers/cor

167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 979us/step - accuracy: 0.8806 - loss: 0.2809
Epoch 100/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7935 - loss: 0.47180
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 905us/step - accuracy: 0.8815 - loss: 0.2898
Epoch 49/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  cy: 0.9062 - loss: 0.17oss: 0.4529
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 603us/step - accuracy: 0.8729 - loss: 0.3029
Epoch 98/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 546us/step - accuracy: 0.8772 - loss: 0.2819
Epoch 100/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 996us/step - accuracy: 0.8768 - loss: 0.2857
Epoch 54/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 929us/step - accuracy: 0.8703 - loss: 0.3093
Epoch 55/100
  1/167 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8750 - loss: 0.2853Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8552 - loss: 0.3533
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8106 - loss: 0.4385
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms

/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/apple/Documents/ai/genai_udemy/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 310us/step - accuracy: 0.6421 - loss: 0.6642
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 275us/step - accuracy: 0.8098 - loss: 0.4484
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 272us/step - accuracy: 0.8139 - loss: 0.4229
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 276us/step - accuracy: 0.8206 - loss: 0.4084
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 283us/step - accuracy: 0.8306 - loss: 0.3944
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 280us/step - accuracy: 0.8380 - loss: 0.3823
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 278us/step - accuracy: 0.8451 - loss: 0.3722
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 285us/step - accuracy: 0.8516 - loss: 0.3652
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 269us/step - accuracy: 0.8534 - loss: 0.3595
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 279us/step - accuracy: 0.8571 - loss: 0.3551
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 275us/step - accuracy: 0.8568 - loss: 0.3520
Epoch 12/100
250/250 ━━━━━━━━━━